# 주행 + 탐지 모델 한 번에 학습 (Colab)

1부는 주행 CNN(`aicar/aicarA.ipynb` 그대로), 2부는 YOLO 탐지, 3부는 결과를 zip 하나로 묶습니다.

1. Colab 파일 창에 두 zip 을 올립니다.
   - `wayA.zip` : Raspberry Pi 의 `dl.py` 에서 [Z] 로 만든 주행 이미지 zip (이름은 프로젝트 이름)
   - `yolo_dataset.zip` : 라벨링한 YOLO 데이터셋 (`images/`, `labels/` 의 `train`·`val`·`test`)
2. 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 저장
3. 런타임 → 모두 실행
4. 끝나면 `<프로젝트>_models.zip` 을 내려받아 Pi 의 `~/aicar/` 에 풉니다 → `ar.py <프로젝트>` 로 주행 + 탐지

세션이 끊겨도 결과가 남도록 1부·2부가 끝날 때마다 Drive(`MyDrive/aicar_models/`)에 복사합니다.


## 0. Drive 연결 (결과 백업용)


In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')
BACKUP_DIR = '/content/drive/MyDrive/aicar_models'
os.makedirs(BACKUP_DIR, exist_ok=True)

def backup(path):
    '''학습 결과를 Drive 로 복사합니다 (파일 또는 폴더).'''
    dst = os.path.join(BACKUP_DIR, os.path.basename(path))
    if os.path.isdir(path):
        shutil.copytree(path, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(path, dst)
    print('Drive 백업:', dst)


# 1부. 주행 CNN 학습 (aicarA.ipynb)


<STEP 1>이미지 파일 압축 풀기 - 실행 하기 전 압축파일(*.zip)을 올려 놓아야 합니다.

In [ ]:
EPOCHS = None; learnRate = None  # epochN = 300, learnRate = 0.0005/0.0010/0.0015/0.0020/0.0025
import glob
import zipfile
fileList = [f for f in glob.glob('*.zip') if f != 'yolo_dataset.zip' and not f.endswith('_models.zip')]
assert len(fileList) == 1, f'주행 이미지 zip 을 하나만 올려 주세요: {fileList}'
zip_ref = zipfile.ZipFile(fileList[0], 'r');
zip_ref.extractall()
zip_ref.close()
w = fileList[0].split('.'); project = w[-2]
print('프로젝트:',project)

<STEP 2> 라이브러리 가져오기(Import)

In [ ]:
import os
import sys
import random
import fnmatch
import pickle
import time
import numpy as np
import cv2                                                                      # Open Cv 영상처리 라이브러리
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchsummary import summary as summary
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import mean_squared_error, r2_score
%matplotlib inline
from PIL import Image

print(f'Python Version:     {sys.version}')                                     # Python Version
print(f'OpenCV version:     {cv2.__version__}')                                  # Open CV version
print(f'Pytorch version:    {torch.__version__}')                               # Pytorch version

<STEP 3> 이미지 데이터 조향각 추출

In [ ]:
with open(f'{project}/_{project}.pickle', 'rb') as f:                           # pickle 파일
    x_exam_Image = pickle.load(f)   # 시험(Test) 이미지 파일 이름 리스트
    x_valid_Image = pickle.load(f)  # 평가(Valid) 이미지 파일 이름 리스트
    x_train_Image = pickle.load(f)  # 훈련(Train) 이미지 파일 이름 리스트
    p = pickle.load(f)              # 파라메터 리스트
if EPOCHS == None: EPOCHS = p[0]    # 학습회수
if learnRate == None: learnRate = p[1]  # 학습 비율
#------------------------------------------------------------------------
y_exam_Angle=[]
y_valid_Angle=[]
y_train_Angle = []
for f in x_exam_Image:
    y_exam_Angle.append(int(f[-7:-4]))  # 시험용 이미지 파일 이름에서 각도값
for f in x_valid_Image:
    y_valid_Angle.append(int(f[-7:-4])) # 평가용 이미지 파일 이름에서 각도값
for f in x_train_Image:
    y_train_Angle.append(int(f[-7:-4])) # 훈련용 이미지 파일 이름에서 각도값

print('학습 회수:',EPOCHS,)
print('학습 비율', learnRate)
print('시험 이미지 개수:',len(x_exam_Image))
print('평가 이미지 개수:',len(x_valid_Image))
print('훈련 이미지 개수:',len(x_train_Image))

<STEP 4> Nvidia CNN 모델 구성

In [ ]:
# Pytorch device 설정
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'{device} is available')

# Nvidia CNN 모델 구성 ---------------------------------------------------------
class NvidiaModel(nn.Module):
    def __init__(self):
      super(NvidiaModel, self).__init__()

      # elu=Expenential Linear Unit, similar to leaky Relu
      # skipping 1st hiddel layer (nomralization layer), as we have normalized the data

      # Convolution Layers
      self.layer1 = nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=24, kernel_size=(5, 5), stride=(2, 2)),
        nn.ELU(inplace=True),
        nn.Conv2d(in_channels=24, out_channels=36, kernel_size=(5, 5), stride=(2, 2)),
        nn.ELU(inplace=True),
        nn.Conv2d(in_channels=36, out_channels=48, kernel_size=(5, 5), stride=(2, 2)),
        nn.ELU(inplace=True),
        nn.Conv2d(in_channels=48, out_channels=64, kernel_size=(3, 3)),
        nn.ELU(inplace=True),
        nn.Dropout(0.2),
        nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3)),
        nn.ELU(inplace=True)
      )

      # Fully Connected Layers
      self.layer2 = nn.Sequential(
        # nn.Flatten(),
        nn.Dropout(0.2),
        nn.Linear(in_features=18 * 64, out_features=100),
        nn.ELU(inplace=True),
        nn.Linear(in_features=100, out_features=50),
        nn.ELU(inplace=True),
        nn.Linear(in_features=50, out_features=10),
        nn.ELU(inplace=True)
      )

      # Output Layer
      self.layer3 = nn.Sequential(
        nn.Linear(in_features=10, out_features=1)
      )

    def forward(self, x):
      x = self.layer1(x)
      x = x.view(x.shape[0], -1)
      x = self.layer2(x)
      x = self.layer3(x)

      return x
#-------------------------------------------------------------------------------
def imgPaths(fGroup):
    r = []
    for filename in fGroup:
        r.append(os.path.join(project, filename))                               # 파일 경로(path)를 화일 이름 앞에 부착하여 리스트에 추가
    return(r)

# 학습 데이터 생성 -------------------------------------------------------------
class CustomDataset(Dataset):
    def __init__(self, imageList, angleList):
        self.imageList = imgPaths(imageList)
        self.angleList = angleList

        for i in range(len(imageList)):
            image = cv2.imread(self.imageList[i])
            image = image / 255
            self.imageList[i] = image

    def __len__(self):
        return len(self.imageList)

    def __getitem__(self, index):
        images = torch.FloatTensor(self.imageList[index]).permute(2,0,1)
        angles = torch.FloatTensor([self.angleList[index]])
        return images, angles
#-------------------------------------------------------------------------------
class EarlyStopping:
    """주어진 patience 이후로 validation loss가 개선되지 않으면 학습을 조기 중지"""
    def __init__(self, patience=7, verbose=False, delta=0, path=f'_{project}_model_check.pt'):
        """
            patience (int): validation loss가 개선된 후 기다리는 기간 - Default: 7
            verbose (bool): True일 경우 각 validation loss의 개선 사항 메세지 출력 - Default: False
            delta (float): 개선되었다고 인정되는 monitered quantity의 최소 변화 - Default: 0
            path (str): checkpoint저장 경로
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'Early Stopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''validation loss가 감소하면 모델을 저장한다.'''
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

<STEP 5> 모델 학습

In [ ]:
# 모델 학습 --------------------------------------------------------------------
def learnProc():
  model = NvidiaModel().to(device)

  train_dataset = CustomDataset(x_train_Image, y_train_Angle)
  train_loader = DataLoader(train_dataset, batch_size=len(x_train_Image))
  valid_dataset = CustomDataset(x_valid_Image, y_valid_Angle)
  valid_loader = DataLoader(valid_dataset, batch_size=len(x_valid_Image))
  optimizer = optim.Adam(model.parameters(), lr=learnRate)
  criterion = nn.MSELoss().to(device)
  early_stopping = EarlyStopping(patience=30, verbose=True)

  train_losses = []
  valid_losses = []
  avg_train_losses = []
  avg_valid_losses = []
  for epoch in range(1, EPOCHS + 1):
      # 학습
      model.train()

      for batch_idx, (data, target) in enumerate(train_loader):
          data, target = data.to(device), target.to(device)
          optimizer.zero_grad()

          output = model(data)

          loss = criterion(output.to(torch.float32), target.to(torch.float32))
          loss.backward()
          optimizer.step()

          train_losses.append(loss.item())

      # 검증
      model.eval()

      #with torch.no_grad():
      with torch.inference_mode():
        for data, target in valid_loader:
            data, target = data.to(device), target.to(device)

            output = model(data)

            loss = criterion(output.to(torch.float32), target.to(torch.float32))

            valid_losses.append(loss.item())

      train_loss = np.average(train_losses)
      valid_loss = np.average(valid_losses)
      avg_train_losses.append(train_loss)
      avg_valid_losses.append(valid_loss)

      epoch_len = len(str(EPOCHS))

      print_msg = (f'[{epoch:>{epoch_len}}/{EPOCHS:>{epoch_len}}] ' +
                      f'train_loss: {train_loss:.6f} ' +
                      f'valid_loss: {valid_loss:.6f}')

      print(print_msg)

      train_losses = []
      valid_losses = []

      early_stopping(valid_loss, model)

      if early_stopping.early_stop:
          print("Early stopping")
          break

  return {'loss':avg_train_losses, 'val_loss':avg_valid_losses}

# 학습 시작---------------------------------------------------------------------
startTime = time.time()                     # 학습 시작 시간 저장
history = learnProc()                       # 학습 시작
elapsedTime = int(time.time() - startTime)  # 학습 경과 시간 저장
lossMin = min(history['loss'])              # history 'loss' 최소값
val_lossMin = min(history['val_loss'])      # history 'val_loss' 최소값
#-------------------------------------------------------------------------------
plt.plot(history['loss'],color='blue')
plt.plot(history['val_loss'],color='red')
plt.legend([f'training loss Min: {lossMin:0.2f}', f'validation loss Min: {val_lossMin:0.2f}'])
plt.savefig(f'_{project}_result.png')                                           # plot 이미지 저장

<STEP 6> 학습 결과 표시, 테스트 파일 각도 예측

In [ ]:
def resultShow():       # 학습 완료후 통계자료를 보여줍니다.
  model = NvidiaModel().to(device)

  # NvidiaModel 의 Summary를 표시합니다.
  summary(model,(3, 66, 200))

  test_dataset = CustomDataset(x_exam_Image, y_exam_Angle)
  test_loader = DataLoader(test_dataset, batch_size=len(x_exam_Image))

  model = NvidiaModel().to(device)
  model.load_state_dict(torch.load(f'_{project}_model_check.pt'))

  # 모델 파일
  model.eval()

  y_pred = []
  y_target = []

  with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            print('pred', output.to(device).cpu().numpy())                                                              # GPU Edition
            print('target', target.to(device).cpu().numpy())                                                            #
            for output_data, target_data in zip(output.to(device).cpu().numpy(), target.to(device).cpu().numpy()):      #
              y_pred.append(output_data)
              y_target.append(target_data)

  mse = mean_squared_error(y_target, y_pred)
  r2s = r2_score(y_target, y_pred)
  #-----------------------------------------------------------------------------
  print('\n표준 편차:', f'{mse:.2f}')
  print('회귀 결정 계수:', f'{r2s:.2%}')
  #-----------------------------------------------------------------------------
  fig, axes = plt.subplots(1, len(x_exam_Image), figsize=(30, 2+3))
  xTImg = imgPaths(x_exam_Image)                                                # 파일 이름 앞에 경로(path) 추가
  for x, c in enumerate(xTImg):
    yuv_image = cv2.imread(c, cv2.IMREAD_UNCHANGED)                             # 테스트 이미지 읽기
    bgr_image = cv2.cvtColor(yuv_image, cv2.COLOR_YUV2BGR)                      # YUV 이미지를 BGR 형식으로 변환
    rgb_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)                      # MatPlot Lib 에서 표시되는 RGB 형식으로 변환
    axes[x].imshow(rgb_image)
    t = x_exam_Image[x] +' / E:'+str(int(y_pred[x]))
    axes[x].set_title(t, fontsize=15)
    axes[x].xaxis.set_ticks([])
    axes[x].yaxis.set_ticks([])

if len(x_exam_Image): resultShow()
print("학습 소요 시간:", int(elapsedTime/60), '분', elapsedTime%60, '초' )      # 학습 소요시간 프린트

In [ ]:
backup(f'_{project}_model_check.pt')   # 주행 모델 백업


# 2부. YOLO 탐지 학습

- 사람: COCO 사전학습 `yolo26n.pt` 를 그대로 씁니다 (학습 없음, 416px 로 NCNN 변환만).
- 노루·멧돼지: `yolo_dataset.zip` 으로 학습합니다 (640px). 클래스 순서는 `yolo/train/data.yaml` 과 같습니다.


In [ ]:
# 2026-11-12 모델과 같은 버전
!pip install -q ultralytics==8.4.211


In [ ]:
import zipfile, yaml
from ultralytics import YOLO

YOLO_EPOCHS = 100      # 데이터 양을 보고 조정
ANIMAL_IMGSZ = 640
PERSON_IMGSZ = 416
YOLO_DIR = '/content/yolo_dataset'

assert os.path.exists('yolo_dataset.zip'), 'yolo_dataset.zip 을 올려 주세요'
with zipfile.ZipFile('yolo_dataset.zip') as z:
    z.extractall(YOLO_DIR)

data = {'path': YOLO_DIR, 'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
        'names': {0: 'person', 1: 'roe_deer', 2: 'wild_boar'}}
for split in ('train', 'val', 'test'):
    n = len(os.listdir(os.path.join(YOLO_DIR, 'images', split)))
    print(f'{split}: {n}장')
    assert n > 0, f'images/{split} 가 비어 있습니다'
with open('animal_data.yaml', 'w') as f:
    yaml.safe_dump(data, f, allow_unicode=True)


In [ ]:
animal = YOLO('yolo26n.pt')
animal.train(data='animal_data.yaml', imgsz=ANIMAL_IMGSZ, epochs=YOLO_EPOCHS, project='/content/runs', name='animal', exist_ok=True)
best = YOLO('/content/runs/animal/weights/best.pt')
backup('/content/runs/animal/weights/best.pt')


In [ ]:
# 학습에 쓰지 않은 test 세트로 평가 → 클래스별 재현율을 yolo/models/README.md 에 기록 (목표 85% 이상)
metrics = best.val(data='animal_data.yaml', imgsz=ANIMAL_IMGSZ, split='test')
for i, name in best.names.items():
    if i in metrics.box.ap_class_index:
        k = list(metrics.box.ap_class_index).index(i)
        print(f'{name:10s} recall={metrics.box.r[k]:.3f}  mAP50={metrics.box.ap50[k]:.3f}')


In [ ]:
# Raspberry Pi 용 NCNN 변환
animal_ncnn = best.export(format='ncnn', imgsz=ANIMAL_IMGSZ)
person_ncnn = YOLO('yolo26n.pt').export(format='ncnn', imgsz=PERSON_IMGSZ)
print(animal_ncnn, person_ncnn)


# 3부. 결과를 zip 하나로 묶기

Pi 의 `~/aicar/` 에서 풀면 `<프로젝트>/` 폴더 안에 주행 모델과 탐지 모델이 함께 놓입니다. `ar.py <프로젝트>` 가 둘 다 찾습니다.


In [ ]:
NCNN_FILES = ('model.ncnn.param', 'model.ncnn.bin', 'metadata.yaml')
out = f'{project}_models.zip'
with zipfile.ZipFile(out, 'w') as z:
    z.write(f'_{project}_model_check.pt', f'{project}/_{project}_model_check.pt')
    for src_dir, name in ((animal_ncnn, 'animal_yolo26n_ncnn'), (person_ncnn, 'person_yolo26n_ncnn')):
        for f in NCNN_FILES:
            z.write(os.path.join(src_dir, f), f'{project}/{name}/{f}')
for n in zipfile.ZipFile(out).namelist():
    print(n)
backup(out)
from google.colab import files
files.download(out)
